In [ ]:
import h5py
import cv2
import numpy as np
import random
from pathlib import Path

In [ ]:
def create_h5_dataset(sample_rate_empty = 0.2, seed = 77): #AK77
    image_folder = Path('preprocessing_output')
    mask_folder = Path('masks_output')
    output_h5 = Path('cell_dataset.h5')

    for folder in [image_folder, mask_folder]:
        if not folder.exists():
            print(f"{folder} not exist")
            return

    random.seed(seed)
    np.random.seed(seed)

    #樣本分類(positive, empty)
    all_image_paths = sorted(list(image_folder.glob('*.png'))) #路徑.glob(檔名) 搜索路徑中所有帶有指定檔名內容的檔案
    positive_samples = []
    empty_samples = []

    for img_path in all_image_paths:
        mask_name = f'{img_path.stem}_mask.png'
        mask_path = mask_folder / mask_name
        if mask_path.exists():
            mask_data = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) #cv2.imread(path, flag) #cv2.IMREAD_GRAYSCALE雙通道灰階 #讀出來會是一個numpy matrix
            if np.sum(mask_data) == 0:
                empty_samples.append(img_path)
            else:
                positive_samples.append(img_path)
    print(f"sorting complete: {len(positive_samples)+len(empty_samples)}")

    num_empty_keep = int(len(positive_samples) * sample_rate_empty)
    num_empty_keep = min(num_empty_keep, len(empty_samples))
    selected_empty = random.sample(empty_samples, num_empty_keep) if empty_samples else [] #random.sample(清單, 數量)不重複的隨機抽取

    valid_samples = positive_samples + selected_empty #合併成最終list

    random.shuffle(valid_samples) #random.shuffle(list)列表中隨機洗牌 #EveryDayImShuffling

    if not valid_samples:
        print(".png file not exist")
        return

    print(f"cell positive: {len(positive_samples)}")
    print(f"cell negative: {len(selected_empty)}")

    #計算索引位置(train:80%, val:10%, test:10%) #找出各組索引的終點位置
    total = len(valid_samples)
    train_end = int(total * 0.8)
    val_end = int(total * 0.9)

    datasets = { #建立dictionary
        'train': valid_samples[:train_end], #key:list[start:end]
        'val': valid_samples[train_end:val_end],
        'test': valid_samples[val_end:]
    }

    with h5py.File(output_h5, 'w') as hf: #h5py.File開啟h5運作模式 #h5py.File(檔案路徑, 操作模式) #'w'寫入, 無檔案則建立檔案, 有檔案直接複寫
        for group_name, file_list in datasets.items(): #.items()專門為dictionary格式去提取key, value
            count = len(file_list) #這個數字要後面作為計算預留空間用
            if count == 0: continue

            #測量img尺寸
            sample_img = cv2.imread(str(file_list[0])) #file_list[0]拿第一張當代表 #cv2.imread(str())改字串格式, cv2預設必須讀自串
            h, w, c = sample_img.shape

            group = hf.create_group(group_name) #建立group ex./train/
            img_ds = group.create_dataset('images', shape=(count, h, w, c), dtype='uint8') #建立dataset ex./train/images
            mask_ds = group.create_dataset('masks', shape=(count, h, w), dtype='uint8')

            for i, img_path in enumerate (file_list): #i:索引數字從0至count-1
                mask_path = mask_folder / f"{img_path.stem}_mask.png"

                img_data = cv2.imread(str(img_path)) #shape = (256, 256, 3)
                mask_data = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) #shape = (256, 256)

                img_ds[i] = img_data #np matrix target [location] = 寫入內容 #numpy語法
                mask_ds[i] = mask_data

                print(img_path.name)
            print(f"{group_name} : {count} data")

    print(f"{output_h5} create complete")

In [ ]:
if __name__ == '__main__':
    create_h5_dataset(sample_rate_empty = 0.2, seed = 77)